# **Telecom Credit Lending (TCL) benchmark**

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Conv1D, Dense, Flatten, Dropout, MaxPooling1D, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import keras_tuner as kt
from larq.layers import QuantConv1D, QuantDense
from larq.models import summary
import larq_compute_engine as lce
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
from tabulate import tabulate

In [ ]:
# Check if GPU is Available
print("Num GPUs Available: ", tf.config.list_physical_devices('GPU'))
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    # Print information about each GPU
    for gpu in gpus:
        print("GPU:", gpu.name)
else:
    print("No GPU devices found.")

## Dataset Preparation

**DISCLAIMER**: The TCL dataset is proprietary to Monty Mobile and cannot be shared due to commercial confidentiality and data protection obligations. To make the pipeline reproducible, this notebook generates a **synthetic dataset with the same structure** as the original: two raw tables (recharge history and lending history) that are converted into 720 timesteps x 6 behavioral features per user, with five credit score classes.

The synthetic data reproduce the format and the processing pipeline, not the results reported in the paper.

Raw table schemas (`TCL_data/`):

| File | Columns |
| --- | --- |
| `charging.csv.gz` | `user_id`, `amount`, `date` |
| `lending.csv.gz` | `user_id`, `amount`, `take_date`, `return_date` |
| `labels.csv.gz` | `user_id`, `label` (0-4, ascending creditworthiness) |

The synthetic files (132,130 users) are provided in `TCL_data/`; if they are missing, the notebook regenerates them with the same seed. To run on real data with the same schema, place the three files in `TCL_data/` and set `USE_SYNTHETIC = False`.

In [ ]:
USE_SYNTHETIC = True
DATA_DIR = 'TCL_data'
SEED = 42

N_DAYS = 90                         # observation window (three months)
BIN_HOURS = 3                       # timestep resolution
T = N_DAYS * 24 // BIN_HOURS        # 720 timesteps
WINDOW_DAYS = 30                    # look-back window for regularity and value features
NUM_CLASSES = 5
START_DATE = pd.Timestamp('2024-01-01')

# Synthetic data only
N_USERS = 132_130                   # as in the original dataset; reduce (e.g., 10_000) for a quicker run
CLASS_PROPS = [0.35, 0.25, 0.18, 0.13, 0.09]   # imbalanced class distribution

### Synthetic data generation

Each synthetic user is assigned a credit score class (0-4) with an imbalanced distribution. The class controls the user's recharge and borrowing behavior (rate, amount and repayment delay), with random per-user variation so that classes overlap.

With `N_USERS = 132_130` (the size of the original dataset), generation takes about 1.5 minutes, the raw CSV files take about 180 MB, and the feature array takes about 2.3 GB of memory.

In [ ]:
def generate_synthetic_tcl(n_users=N_USERS, n_days=N_DAYS, start=START_DATE, seed=SEED):
    rng = np.random.default_rng(seed)
    labels = rng.choice(NUM_CLASSES, size=n_users, p=CLASS_PROPS)

    # Class-dependent behavior profiles, from lowest (0) to highest (4) creditworthiness
    recharge_rate = np.array([0.15, 0.25, 0.40, 0.60, 0.90])     # recharges per day
    recharge_amount = np.array([1.0, 1.5, 2.5, 4.0, 6.0])        # mean recharge amount
    loan_rate = np.array([0.03, 0.06, 0.08, 0.10, 0.12])         # loans per day
    loan_amount = np.array([0.5, 1.0, 2.0, 3.5, 5.0])            # mean loan amount
    repay_days = np.array([12.0, 7.0, 4.0, 2.5, 1.5])            # mean repayment delay
    unpaid_prob = np.array([0.30, 0.15, 0.07, 0.03, 0.01])       # share of loans never repaid

    horizon_s = n_days * 24 * 3600
    charging, lending = [], []
    for uid, c in enumerate(labels):
        # Per-user variation makes neighboring classes overlap
        scale = rng.lognormal(0.0, 0.35)

        n_rech = rng.poisson(recharge_rate[c] * scale * n_days)
        t_rech = np.sort(rng.integers(0, horizon_s, n_rech))
        amt_rech = rng.lognormal(np.log(recharge_amount[c] * scale), 0.5, n_rech)
        charging.append(pd.DataFrame({'user_id': uid,
                                      'amount': amt_rech.round(2),
                                      'date': start + pd.to_timedelta(t_rech, unit='s')}))

        n_loan = rng.poisson(loan_rate[c] * scale * n_days)
        t_loan = np.sort(rng.integers(0, horizon_s, n_loan))
        amt_loan = rng.lognormal(np.log(loan_amount[c]), 0.3, n_loan)
        delay_s = rng.exponential(repay_days[c] * 24 * 3600, n_loan).round()
        take = start + pd.to_timedelta(t_loan, unit='s')
        ret = take + pd.to_timedelta(delay_s, unit='s')
        ret = ret.where(rng.random(n_loan) >= unpaid_prob[c], pd.NaT)
        lending.append(pd.DataFrame({'user_id': uid,
                                     'amount': amt_loan.round(2),
                                     'take_date': take,
                                     'return_date': ret}))

    charging = pd.concat(charging, ignore_index=True)
    lending = pd.concat(lending, ignore_index=True)
    labels = pd.DataFrame({'user_id': np.arange(n_users), 'label': labels})
    return charging, lending, labels

In [ ]:
if USE_SYNTHETIC and not os.path.exists(f'{DATA_DIR}/charging.csv.gz'):
    charging, lending, labels = generate_synthetic_tcl()
    os.makedirs(DATA_DIR, exist_ok=True)
    charging.to_csv(f'{DATA_DIR}/charging.csv.gz', index=False)
    lending.to_csv(f'{DATA_DIR}/lending.csv.gz', index=False)
    labels.to_csv(f'{DATA_DIR}/labels.csv.gz', index=False)

charging = pd.read_csv(f'{DATA_DIR}/charging.csv.gz', parse_dates=['date'])
lending = pd.read_csv(f'{DATA_DIR}/lending.csv.gz', parse_dates=['take_date', 'return_date'])
labels = pd.read_csv(f'{DATA_DIR}/labels.csv.gz')

print(charging.head())
print(lending.head())
print(labels['label'].value_counts().sort_index())

### Feature extraction

The raw records are condensed into six behavioral features per user, computed at each of the 720 timesteps:

| Feature | Definition |
| --- | --- |
| Recharge Recency | hours since the last recharge |
| Recharge Regularity | number of recharges in the last `WINDOW_DAYS` days |
| Recharge Value | total recharged amount in the last `WINDOW_DAYS` days |
| Borrowing Recency | hours since the last loan |
| Borrowing Regularity | number of loans in the last `WINDOW_DAYS` days |
| Credit Exposure | total borrowed amount in the last `WINDOW_DAYS` days |

Before the first event of a user, recency is set to the full observation window (`N_DAYS * 24` hours).

In [ ]:
def event_bins(df, time_col, user_index):
    # Map each event to (user row, timestep) and drop events outside the observation window
    rows = df['user_id'].map(user_index).to_numpy()
    bins = ((df[time_col] - START_DATE) // pd.Timedelta(hours=BIN_HOURS)).to_numpy()
    valid = (bins >= 0) & (bins < T) & ~pd.isna(rows)
    return rows[valid].astype(int), bins[valid].astype(int), df['amount'].to_numpy()[valid]


def rfm_features(df, time_col, user_index, n_users):
    rows, bins, amounts = event_bins(df, time_col, user_index)
    counts = np.zeros((n_users, T), dtype=np.float32)
    values = np.zeros((n_users, T), dtype=np.float32)
    np.add.at(counts, (rows, bins), 1)
    np.add.at(values, (rows, bins), amounts)

    # Recency: hours since the last timestep with an event
    steps = np.arange(T)
    last = np.where(counts > 0, steps, -1)
    last = np.maximum.accumulate(last, axis=1)
    recency = np.where(last >= 0, (steps - last) * BIN_HOURS, N_DAYS * 24).astype(np.float32)

    # Regularity and value: sums over a trailing window
    w = WINDOW_DAYS * 24 // BIN_HOURS
    def trailing_sum(x):
        c = np.cumsum(x, axis=1)
        c[:, w:] = c[:, w:] - c[:, :-w]
        return c

    return recency, trailing_sum(counts), trailing_sum(values)


def build_features(charging, lending, labels):
    user_ids = labels['user_id'].to_numpy()
    user_index = pd.Series(np.arange(len(user_ids)), index=user_ids)
    rech = rfm_features(charging, 'date', user_index, len(user_ids))
    loan = rfm_features(lending, 'take_date', user_index, len(user_ids))
    # (users, 720, 6): recharge recency/regularity/value, borrowing recency/regularity, credit exposure
    X = np.stack(rech + loan, axis=-1)
    y = labels['label'].to_numpy()
    return X, y


feature_names = ['Recharge Recency', 'Recharge Regularity', 'Recharge Value',
                 'Borrowing Recency', 'Borrowing Regularity', 'Credit Exposure']
X_all, y_all = build_features(charging, lending, labels)
print(X_all.shape, y_all.shape)

In [ ]:
def dataset(X, y, seed=SEED):
    # User-level stratified split: 70% train, 10% validation, 20% test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=1/5, stratify=y, random_state=seed)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=1/8, stratify=y_train, random_state=seed)

    # Standardize each feature with statistics from the training set only
    n_feat = X.shape[-1]
    scaler = StandardScaler()
    scaler.fit(X_train.reshape(-1, n_feat))
    X_train = scaler.transform(X_train.reshape(-1, n_feat)).reshape(X_train.shape)
    X_val = scaler.transform(X_val.reshape(-1, n_feat)).reshape(X_val.shape)
    X_test = scaler.transform(X_test.reshape(-1, n_feat)).reshape(X_test.shape)

    print(X_train.shape, X_val.shape, X_test.shape)

    X_train = tf.convert_to_tensor(X_train, dtype=tf.float32)
    y_train = tf.convert_to_tensor(y_train, dtype=tf.float32)
    X_val = tf.convert_to_tensor(X_val, dtype=tf.float32)
    y_val = tf.convert_to_tensor(y_val, dtype=tf.float32)
    X_test = tf.convert_to_tensor(X_test, dtype=tf.float32)
    y_test = tf.convert_to_tensor(y_test, dtype=tf.float32)

    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
classes = ['0', '1', '2', '3', '4']
X_train, X_val, X_test, y_train, y_val, y_test = dataset(X_all, y_all)
width, n_features = X_train.shape[1], X_train.shape[2]

## NN Evaluation Methods

In [ ]:
def evaluate_NN_models(model_list, X_test, y_test, bin_class=False):
    loss, accuracy, y_pred, precision, recall, f1_score, support, confusion_matrix_list = [], [], [], [], [], [], [], []
    for model in model_list:
        loss_value, accuracy_value = model.evaluate(X_test, y_test)
        loss.append(loss_value)
        accuracy.append(accuracy_value)
        if bin_class:
            y_pred_value = (model.predict(X_test) > 0.5).astype(int)
        else:
            y_pred_value = np.argmax(model.predict(X_test), axis=1)
        y_pred.append(y_pred_value)
        y_test_classes = y_test
        precision_value, recall_value, f1_score_value, support_value = precision_recall_fscore_support(
            y_test_classes,
            y_pred_value,
            average='binary' if bin_class else 'weighted',
            zero_division=0
        )
        precision.append(precision_value)
        recall.append(recall_value)
        f1_score.append(f1_score_value)
        support.append(support_value)
        confusion_matrix_value = confusion_matrix(y_test_classes, y_pred_value)
        confusion_matrix_list.append(confusion_matrix_value)
    return [accuracy, precision, recall, f1_score], confusion_matrix_list, y_pred

In [ ]:
def print_table(data,headers):
    table_data = list(zip(*data))
    table = tabulate(table_data, headers=headers, tablefmt='grid')
    print(table)

In [ ]:
def plot_confusion_matrix(ax, conf_matrix, title, cmap):
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap=cmap, cbar=False, ax=ax)
    ax.set_title(title)

In [ ]:
def draw_confusion_matrix(matrix,name):
    colormap=["Blues","Greens","Oranges"]
    fig, axes = plt.subplots(1, 1, figsize=(15, 5))
    for i in range(len(matrix)):
        plot_confusion_matrix(axes, matrix[i], "Confusion Matrix  ", colormap[i])
    plt.tight_layout()

In [ ]:
# Define early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    verbose=1,
    mode='min',
    restore_best_weights=True
)

## Hyperparameters Optimization

The BNN used in the paper for TCL (Fig. 3) is defined below in `paper_hps`. Set `RUN_TUNER = True` to search the hyperparameters again with Keras Tuner (20 trials of 50 epochs, early stopping patience 15). The search covers the seven BNNForge hyperparameters, with ranges extended with respect to the defaults (Table 1) so that they include the paper architecture; kernel sizes are fixed to those in Fig. 3.

In [ ]:
RUN_TUNER = False

# TCL architecture reported in the paper (Fig. 3)
paper_hps = {
    'conv1_filters': 64, 'conv1_kernel': 3,
    'qconv1_filters': 32, 'qconv1_kernel': 2, 'dropout1': 0.5,
    'qconv2_filters': 64, 'qconv2_kernel': 3, 'dropout2': 0.4,
    'qdense_units': 64,
    'learning_rate': 1e-3,
}

In [ ]:
def build_bnn(hps, input_shape, num_classes):
    # All quantized layers except the first will use the same options
    kwargs = dict(input_quantizer="ste_sign",
                  kernel_quantizer="ste_sign",
                  kernel_constraint="weight_clip")

    model = Sequential()
    # First Conv1D layer (full precision)
    model.add(Conv1D(filters=hps['conv1_filters'], kernel_size=hps['conv1_kernel'], padding='same',
                     activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(BatchNormalization(scale=False))

    # First QuantConv1D layer
    model.add(QuantConv1D(hps['qconv1_filters'], kernel_size=hps['qconv1_kernel'], padding='same',
                          kernel_quantizer="ste_sign",
                          kernel_constraint="weight_clip",
                          use_bias=False))
    model.add(MaxPooling1D(pool_size=2))
    model.add(BatchNormalization(scale=False))
    model.add(Dropout(rate=hps['dropout1']))

    # Second QuantConv1D layer
    model.add(QuantConv1D(hps['qconv2_filters'], kernel_size=hps['qconv2_kernel'], padding='same',
                          use_bias=False, **kwargs))
    model.add(MaxPooling1D(pool_size=2))
    model.add(BatchNormalization(scale=False))
    model.add(Dropout(rate=hps['dropout2']))

    model.add(Flatten())

    # QuantDense layer
    model.add(QuantDense(hps['qdense_units'], use_bias=False, **kwargs))
    model.add(BatchNormalization(scale=False))
    model.add(Dense(num_classes, activation='softmax'))
    return model

In [ ]:
class BinaryNetHyperModel(kt.HyperModel):
    def __init__(self, input_shape, num_classes):
        self.input_shape = input_shape
        self.num_classes = num_classes

    def build(self, hp):
        hps = {
            'conv1_filters': hp.Int('conv1_filters', min_value=16, max_value=64, step=16),
            'conv1_kernel': 3,
            'qconv1_filters': hp.Int('qconv1_filters', min_value=8, max_value=64, step=8),
            'qconv1_kernel': 2,
            'dropout1': hp.Float('dropout1', min_value=0.2, max_value=0.5, step=0.1),
            'qconv2_filters': hp.Int('qconv2_filters', min_value=8, max_value=64, step=8),
            'qconv2_kernel': 3,
            'dropout2': hp.Float('dropout2', min_value=0.2, max_value=0.5, step=0.1),
            'qdense_units': hp.Int('qdense_units', min_value=16, max_value=64, step=16),
        }
        model = build_bnn(hps, self.input_shape, self.num_classes)

        # Tune learning rate
        learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')
        model.compile(
            optimizer=Adam(learning_rate=learning_rate),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        return model

In [ ]:
if RUN_TUNER:
    # Initialize the hypermodel
    hypermodel = BinaryNetHyperModel(input_shape=(width, n_features), num_classes=len(classes))

    # Create the tuner
    tuner = kt.BayesianOptimization(
        hypermodel,
        objective='val_accuracy',
        max_trials=20,
        directory='bayesian_opt',
        project_name='tcl_tuning'
    )

    # Search for the best hyperparameters
    tuner.search(
        X_train,
        y_train,
        epochs=50,
        batch_size=64,
        validation_data=(X_val, y_val),
        callbacks=[early_stopping]
    )

    # Get the best hyperparameters
    best_hps_list = tuner.get_best_hyperparameters(num_trials=3)
    best_models_list = tuner.get_best_models(num_models=3)

In [ ]:
if RUN_TUNER:
    # Show the top 3 models
    print("\nTop 3 Models Performance:")
    print("-" * 50)

    for idx, hps in enumerate(best_hps_list, 1):
        print(f"\nModel {idx}:")
        print("Hyperparameters:")
        for param, value in hps.values.items():
            print(f"{param}: {value}")
        print(f"Validation Accuracy: {best_models_list[idx-1].evaluate(X_val, y_val, verbose=0)[1]:.3f}")

    # Let user choose the model
    while True:
        try:
            choice = int(input("\nChoose a model (1, 2, or 3): "))
            if 1 <= choice <= 3:
                break
            print("Please enter 1, 2, or 3")
        except ValueError:
            print("Please enter a valid number")

    # Use the chosen model
    chosen_hps = dict(paper_hps, **best_hps_list[choice - 1].values)
    print(f"\nSelected Model {choice}")
else:
    chosen_hps = paper_hps

print("Final hyperparameters:")
for param, value in chosen_hps.items():
    print(f"{param}: {value}")

## Build BNN

In [ ]:
model = build_bnn(chosen_hps, input_shape=(width, n_features), num_classes=len(classes))

In [ ]:
lr = chosen_hps['learning_rate']
epochs = 300
batch_size = 64

In [ ]:
model.compile(optimizer=Adam(learning_rate=lr), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print(summary(model))

### Train

As in the paper, the selected model is retrained on the union of the training and validation sets.

In [ ]:
X_trainval = tf.concat([X_train, X_val], axis=0)
y_trainval = tf.concat([y_train, y_val], axis=0)

# No validation set is left, so early stopping monitors the training loss
early_stopping_final = EarlyStopping(
    monitor='loss',
    patience=15,
    verbose=1,
    mode='min',
    restore_best_weights=True
)

history= model.fit(X_trainval, y_trainval, epochs=epochs, batch_size=batch_size, shuffle=True, callbacks=[early_stopping_final], verbose=1)

### Evaluate

In [ ]:
result_all_models_FC, confusion_matrix_FC, predicitions_FC = evaluate_NN_models([model], X_test, y_test, bin_class=False)

In [ ]:
headers=['models','accuracy','precision','recall','f1_score']
data_normal_FC=[["model"]]
data_normal_FC.extend(result_all_models_FC)
print_table(data_normal_FC,headers)

In [ ]:
draw_confusion_matrix(confusion_matrix_FC,"Confusion Matrix")

In [ ]:
plt.figure()
plt.plot(history.history['accuracy'], color='orange', label='Training Accuracy')
plt.xlabel("epochs")
plt.ylabel("accuracy")
plt.show()

## Save

In [ ]:
# Convert our Keras model to a TFLite flatbuffer file
with open("TCL_model.tflite", "wb") as flatbuffer_file:
    flatbuffer_bytes = lce.convert_keras_model(model)
    flatbuffer_file.write(flatbuffer_bytes)

## Convert Model to C using CBin-NN

In [ ]:
# Save the trained Keras model to .h5
MODEL_H5_NAME = "TCL_model.h5"
model.save(MODEL_H5_NAME)
print(f"Saved Keras model to {MODEL_H5_NAME}")

In [ ]:
# Clone the CBin-NN repo
if not os.path.exists("CBin-NN"):
    !git clone https://github.com/Edge-Learning-Machine/CBin-NN.git
else:
    print("CBin-NN already cloned.")

In [ ]:
# Copy the trained Keras model into the CBin-NN Code Generation folder,
# where model_converter.py expects to find it
import shutil

CODE_GEN_DIR = os.path.join("CBin-NN", "Code Generation")
shutil.copy(MODEL_H5_NAME, os.path.join(CODE_GEN_DIR, MODEL_H5_NAME))
print(f"Copied {MODEL_H5_NAME} to {CODE_GEN_DIR}")

In [ ]:
# Update model_converter.py so model_name points to our trained model
converter_path = os.path.join(CODE_GEN_DIR, "model_converter.py")

with open(converter_path, "r") as f:
    converter_src = f.read()

converter_src = converter_src.replace(
    "model_name = 'lenet_PReLU.h5'",
    f"model_name = '{MODEL_H5_NAME}'"
)

with open(converter_path, "w") as f:
    f.write(converter_src)

print(f"Updated model_name to '{MODEL_H5_NAME}' in {converter_path}")

In [ ]:
# Run model_converter.py from inside the Code Generation folder to generate the C code
!cd "{CODE_GEN_DIR}" && python model_converter.py